<a href="https://colab.research.google.com/github/Kalyani-asekar/NNDL/blob/main/practical6nndl_10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

torch.manual_seed(42)
seq_len = 10
input_dim = 1
num_samples = 1000

t = torch.linspace(0, 50, seq_len)
data = torch.sin(t).unsqueeze(0).repeat(num_samples, 1).unsqueeze(-1)
data += torch.randn_like(data) * 0.1
data = data.to(device)

class Encoder(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super(Encoder, self).__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, batch_first=True)

    def forward(self, x):
        _, (hn, _) = self.lstm(x)
        return hn.squeeze(0)

class Decoder(nn.Module):
    def __init__(self, hidden_dim, output_dim, seq_len):
        super(Decoder, self).__init__()
        self.seq_len = seq_len
        self.lstm = nn.LSTM(hidden_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        x = x.unsqueeze(1).repeat(1, self.seq_len, 1)
        out, _ = self.lstm(x)
        out = self.fc(out)
        return out

class LSTMAutoencoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, seq_len):
        super(LSTMAutoencoder, self).__init__()
        self.encoder = Encoder(input_dim, hidden_dim)
        self.decoder = Decoder(hidden_dim, input_dim, seq_len)

    def forward(self, x):
        latent = self.encoder(x)
        reconstruction = self.decoder(latent)
        return reconstruction

hidden_dim = 16
model = LSTMAutoencoder(input_dim, hidden_dim, seq_len).to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

epochs = 100
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    outputs = model(data)
    loss = criterion(outputs, data)
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 20 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.6f}")

model.eval()
with torch.no_grad():
    sample_input = data[:1]
    reconstructed = model(sample_input)
    print("\nOriginal Sample:\n", sample_input.squeeze().cpu().numpy())
    print("\nReconstructed Sample:\n", reconstructed.squeeze().cpu().numpy())